<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor experiment

## Why this problem?

A committor is the probability that a stochastic trajectory reaches a target set before it returns to a source set. It is a central quantity in transition path theory and metastability: it describes which way a transition is likely to go from each starting point. This example tests a boundary-adapted neural representation on a small periodic problem with two separate interior boundaries.

This is a **committor** experiment, not a mean escape time experiment. The committor answers a probability question; a mean escape time answers how long a process typically takes to hit a set. The mathematical motivation for boundary-adapted PINNs is discussed in the accompanying paper, [*Boundary-Adapted PINNs for Elliptic Dirichlet Problems*](https://arxiv.org/abs/2607.19167).

## Stochastic dynamics and the committor equation

Let $X_t$ follow overdamped Langevin dynamics on the flat torus $\mathbb{T}^2=[0,1)^2$:
$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t\pmod 1,
\qquad
V(x_1,x_2)=a[1-\cos(4\pi(x_1-1/4))]+b[1-\cos(2\pi x_2)].
$$
The periodic disks $A$ and $B$ have centers $c_A=(1/4,0)$ and $c_B=(3/4,0)$ and common radius $r$. For their first hitting times $\tau_A$ and $\tau_B$, the forward committor is
$$
q(x)=\mathbb{P}_x(\tau_B<\tau_A).
$$
It is the probability of reaching $B$ before $A$ when starting at $x$. The diffusion generator is
$$
\mathcal{L}f=-\nabla V\cdot\nabla f+\varepsilon\Delta f.
$$
The Markov property implies that $q$ is harmonic with respect to this generator:
$$
\mathcal{L}q=0\quad\text{in }D=\mathbb{T}^2\setminus(A\cup B),
\qquad q|_{\partial A}=0,\quad q|_{\partial B}=1.
$$
The default parameters are $a=1$, $b=0.6$, and $\varepsilon=0.4$.

## Exact boundary values from a smooth geometry factor

For a periodic disk centered at $c_j$, the code uses the smooth periodic defining function
$$
\omega_j(x)=4\sum_{\ell=1}^2\sin^2(\pi(x_\ell-c_{j\ell}))-r^2.
$$
It is negative inside the disk, zero on its boundary, and positive in the exterior domain. The normalized defining function is
$$
d_j(x)=\frac{\omega_j(x)}{\sqrt{\omega_j(x)^2+|\nabla\omega_j(x)|^2}}.
$$
On the boundary, $d_j=0$ and $|\nabla d_j|=1$. In $D$, both $d_A$ and $d_B$ are positive. Define the boundary lift $g$ and the two-boundary factor $\rho$ by
$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},\qquad
\rho(x)=\frac{d_A(x)d_B(x)}{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$
The network represents
$$
q_\theta(x)=g(x)+\rho(x)N_\theta(x).
$$
On $\partial A$, $g=0$ and $\rho=0$; on $\partial B$, $g=1$ and $\rho=0$. Therefore the two Dirichlet values hold for every choice of network weights. Near either boundary, $\rho$ has the first-order scaling of that boundary's normalized defining function. The simpler product $d_Ad_B$ also vanishes on both boundaries, but its normal scaling depends on the distance to the other boundary.

All three networks receive sine and cosine coordinates $(\sin(2\pi x_1),\sin(2\pi x_2),\cos(2\pi x_1),\cos(2\pi x_2))$, so their outputs are periodic by construction.

## Methods and numerical comparison

The experiment compares three representations:

1. **Boundary-adapted:** exact lift $g$ and normalized factor $\rho$.
2. **Naïve product:** the same lift with the unnormalized product $d_Ad_B$ in place of $\rho$.
3. **Penalty:** a periodic multilayer perceptron trained with sampled boundary penalty terms.

A finite-difference solution on periodic grids provides a numerical reference. Comparing two grid resolutions helps estimate reference discretization error; it is a numerical check, not exact ground truth. The saved figures show the three learned committors, the reference, and pointwise errors. The summary reports grid-based errors and boundary diagnostics.

## Further reading

- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- Y. Gao, T. Li, X. Li, and J.-G. Liu, “Transition path theory for Langevin dynamics on manifold: optimal control and data-driven solver,” [arXiv:2010.09988](https://arxiv.org/abs/2010.09988) (2020).

## What each stage does

| Stage | Purpose | Compute | Persistent output |
|---|---|---|---|
| Setup and CPU tests | Fetches the selected code, installs it, and runs mathematical and tiny training tests. | CPU | None; test files are temporary. It prints the code revision. |
| CPU preflight | Runs the whole comparison with tiny networks and grids to catch wiring errors. | CPU | Temporary Colab files only. It does not use Drive or GPU. |
| GPU pilot | Trains the three configured methods for 6,000 epochs each and computes reference solutions. | GPU for training; CPU for finite differences | A timestamped run folder in Google Drive. |
| Inspect | Reads the saved summary and displays the loss and solution plots. | CPU/display only | No new training and no new results folder. |
| Boundary audit | Recomputes boundary errors from saved best checkpoints. | CPU only | Adds `boundary_diagnostics.json`; does not retrain or replace `summary.json`. |
| Resume | Continues an interrupted run from its Drive checkpoints. | GPU | Writes into the same run folder. |

## Where results go

The GPU cell prints a path beginning with `/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/`. That complete folder is the durable record of the run. Keep it in Drive. The notebook and its displayed plots are not the results archive.

Inside the run folder:

- `summary.json` contains the solution errors and finite-difference grid comparison.
- `comparison_config.json` records the exact problem, model, training settings, and methods.
- `comparison_metadata.json` records the run time and code revision for runs made with the updated runner.
- `reference_fd_256.npz` contains the reference grid, values, and domain masks.
- Each method folder contains `metrics.jsonl` (epoch, objective, PDE loss, boundary loss, and learning rate), `best.pt` (best model weights), and `last.pt` (restart checkpoint and optimizer state).
- `committor_*.png` and `losses.png` are quick diagnostics. They are useful for inspection, but are not publication figures.

The three objective curves are not directly comparable in height: the penalty method adds a weighted boundary penalty, while the two exact-boundary methods train on the PDE residual. Compare `grid_l2` and `grid_h1_seminorm` in `summary.json` to assess agreement with the finite-difference solution, and examine `reference_convergence` to see how much that reference changes from the 128 grid to the 256 grid. The finite-difference solution is a numerical reference, not exact ground truth.

Keep full checkpoints and arrays in Drive. Do not commit the output notebook or large run folder to GitHub. For a paper result, repeat across seeds, check reference-grid convergence, and then add only selected small figures/tables together with the configuration and code revision to the repository.

### GitHub token

While the repository is private, the setup cell uses your saved `GITHUB_TOKEN` Colab Secret to fetch the latest code from the repository's `main` branch. It refreshes only the temporary clone under `/content/boundary-adapted-pinns`. The token grants repository read access; the numerical code does not use it, and it is not saved to Drive or to the repository. You already created it for this private-repository fetch, so keep it in Colab Secrets while the repository is private. It uses no GPU time. If the repository becomes public, setup can fetch the code without the token.

## 1. Setup and CPU tests — CPU only

Run this cell first. It fetches the current repository into Colab's `/content` directory, installs the Python package, and runs the tests. One test runs a tiny one-epoch CPU comparison through all three methods and checks its reference, checkpoints, and summary. The test uses temporary Colab files and no GPU or Google Drive storage. The printed code revision identifies the code used for subsequent runs.

In [ ]:
# CPU setup: fetch main, install the package, and run tests including a tiny CPU training check.
from pathlib import Path
import base64
import os
import subprocess

from google.colab import userdata

CPU_PREFLIGHT_PASSED = False

def run_logged(command, log_path, append=False):
    """Show output and offer the complete log as a download if the command fails."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a" if append else "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        print(f"\nCommand failed (exit {return_code}). The full traceback is above.")
        try:
            from google.colab import files
            files.download(str(log_path))
            print("Attach the downloaded console.log if you need help diagnosing the error.")
        except Exception as download_error:
            print(f"Log download did not start: {download_error}. Copy the traceback above.")
        raise RuntimeError("Command failed. The traceback and downloadable log are above.")

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Public repositories can be cloned without a Colab Secret.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch main from GitHub. If this repository is private, add the GITHUB_TOKEN '
        'Colab Secret and enable Notebook access. The token is only used to read the repository.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Setup and CPU tests passed. Code revision:', revision)


## 2. CPU preflight — full pipeline check

This two-epoch check runs entirely on the CPU and stores its reference, checkpoints, figures, and log in Colab's temporary `/content` storage. Repeating the cell replaces the previous temporary preflight. No preflight files are written to Google Drive. If the command fails, its traceback appears here and Colab offers `console.log` as a browser download. The GPU pilot and resume cells stay blocked until this check succeeds in the current runtime.

In [ ]:
# CPU only: run the full pipeline in temporary Colab storage.
from pathlib import Path
import shutil

CPU_PREFLIGHT_PASSED = False
preflight_dir = Path('/content/periodic_committor_preflight')
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir()
PREFLIGHT_ROOT = str(preflight_dir)
print('CPU preflight outputs (temporary Colab storage):', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], preflight_dir / 'console.log')
if not (preflight_dir / 'summary.json').is_file():
    raise RuntimeError('CPU preflight produced no summary.json; check the command output above.')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. The GPU pilot is now available in this runtime.')

## 3. GPU pilot — first substantive numerical experiment

The GPU cell runs only after the CPU preflight succeeds in this runtime. It uses 64-wide, 4-hidden-layer networks; 1,024 interior points; 6,000 epochs per method; and 128/256-grid references. It mounts Google Drive when started, then saves restart checkpoints and its console output there. On failure, the log is offered as a browser download.

In [ ]:
# GPU required: full three-method committor pilot.
if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('GPU pilot blocked: the CPU preflight has not passed in this runtime. Fix its error first.')
from datetime import datetime
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The GPU pilot requires a T4 GPU. Select it under Runtime → Change runtime type, then rerun setup and preflight.')

from google.colab import drive
drive.mount('/content/drive')

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('GPU pilot results:', RUN_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
], Path(RUN_ROOT) / 'console.log')

## 4. Inspect the finished GPU-pilot outputs

Run this after the GPU pilot cell finishes. It reads `summary.json` and displays the saved training-loss and learned-solution plots. It performs no training. The images are diagnostic previews; the numerical quantities and reference arrays are saved separately in the Drive run folder described above.

In the summary, use `grid_l2` and `grid_h1_seminorm` for the learned-solution comparison, and `reference_convergence` for the finite-difference grid check. The current pilot's `source_boundary_rmse` and `target_boundary_rmse` fields were computed on interior grid nodes by an earlier version of the diagnostic code. Do not use those two fields; the CPU audit in Step 5 recalculates them from actual boundary points.

The loss plot shows each method's training objective. The penalty method includes a weighted boundary penalty in that objective; the exact-boundary methods do not. Compare the curves as optimization histories, not as a ranking by their final vertical values.

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## 5. Check boundary values using the saved models — CPU only

This optional check is for a completed GPU pilot. It loads each method's `best.pt` checkpoint on the CPU, samples points on the actual source and target boundaries, and writes `boundary_diagnostics.json` inside the existing Drive run folder. It does not train, use a GPU, or overwrite `summary.json`.

After this code update is merged, start a fresh Colab runtime and run Step 1 to install the updated code. Then run this cell. If `RUN_ROOT` is not already set in the notebook, it asks for the exact Drive folder path printed by the GPU pilot.

In [ ]:
# CPU only: evaluate saved best.pt checkpoints on the true boundaries; no training occurs.
from pathlib import Path
import json
import subprocess
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'comparison_config.json').is_file():
    RUN_ROOT = input('Paste the completed GPU pilot Drive folder path: ').strip()

if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError(f'No comparison_config.json found in {RUN_ROOT}')

metadata_path = Path(RUN_ROOT) / 'comparison_metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
training_revision = metadata.get('git_revision')
if not training_revision:
    training_revision = input(
        'This older run did not save its code revision. In the Setup cell output, '
        'copy the 40-character SHA after "Code revision:" and paste it here. '
        'Leave blank if you cannot find it: '
    ).strip()

command = ['periodic-committor', '--output-dir', RUN_ROOT, '--audit-boundaries']
if training_revision:
    command += ['--training-git-revision', training_revision]
subprocess.run(command, check=True)

diagnostics_path = Path(RUN_ROOT) / 'boundary_diagnostics.json'
print(diagnostics_path.read_text(encoding='utf-8'))


## Resume an interrupted GPU pilot

Use this only if the GPU pilot started and then stopped before finishing. In a fresh runtime, run setup and pass the CPU preflight first. Then paste the Drive folder path printed by the interrupted pilot. The saved checkpoints continue the existing run. The resume cell is locked until the CPU preflight passes.

In [ ]:
# Run this only if the GPU pilot stopped before it finished.
if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('Resume blocked: rerun setup and pass the CPU preflight in this runtime first.')
# Paste the exact Drive folder path printed at the start of that pilot.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

RUN_ROOT = input('Paste the interrupted pilot Drive folder path, then press Enter: ').strip()
if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError('That folder is missing comparison_config.json. Paste the exact folder path printed by the interrupted pilot.')

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--resume',
], Path(RUN_ROOT) / 'console.log', append=True)
print('The interrupted pilot has resumed and completed.')

## One-time Drive cleanup (optional)

Earlier notebook versions saved CPU checks and some failed GPU attempts to Drive. Run the next cell once, while no experiment is running, to remove the timestamped CPU preflight folders. It also removes a GPU attempt folder only if it contains nothing beyond a console log and run metadata. Any GPU folder containing reference data, checkpoints, figures, or other files is preserved. Current and future CPU preflights stay in temporary Colab storage.

In [ ]:
# Optional: remove old CPU checks and empty failed GPU attempts created by this notebook.
from pathlib import Path
import re
import shutil
from google.colab import drive

drive.mount('/content/drive')
old_run_root = Path('/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor')
known_startup_files = {'console.log', 'comparison_config.json', 'comparison_metadata.json'}
old_cleanup_targets = []
if old_run_root.is_dir():
    for folder in old_run_root.iterdir():
        if not folder.is_dir():
            continue
        if re.fullmatch(r'preflight_\d{8}_\d{6}', folder.name):
            old_cleanup_targets.append(folder)
        elif re.fullmatch(r'periodic_committor_\d{8}_\d{6}', folder.name):
            if all(child.is_file() and child.name in known_startup_files for child in folder.iterdir()):
                old_cleanup_targets.append(folder)
for folder in sorted(old_cleanup_targets):
    shutil.rmtree(folder)
    print('Removed:', folder.name)
print(f'Drive cleanup complete: {len(old_cleanup_targets)} old folders removed.')